# Regress Thermodynamic Data using PropFit  

Author: Jordyn Robare  
email: jordynrobare@gmail.com  

This notebook is made to demonstrate how to regress thermodynamic data for organic molecules in order to estimate group contribution values. 
Group contribution values are then made into files that can be used in the AqOrg package (Boyer et al., 2025) to estimate properties of whole molecules.

### 1. Import necessary packages

In [ ]:
from aqorg import Estimate
import pandas as pd
import propfit
from pychnosz import add_OBIGT, info

### 2. Initialize the package using the Propfit class  
For the thermodynamic database, we provide a CSV named 'default database.csv' that may be sufficient. However, it can be modified into a custom database supplied by the user.   
Provide the names of the thermodynamic properties that should be regressed unless they match the default properties provided.  
If you plan on using a custom group-matching sheet, provide that here.  

* filename= None (uses 'default database.csv' if None provided. Otherwise, provide the name of a CSV)
* props=['Gh','Hh','Cph','V','Hig','Sig','Cpig'] (properties that will be regressed. Need to match columns in database)
* group_file=None (optional user-supplied group definitions)

Refer to these resources for help coding SMARTS groups:  
https://www.daylight.com/dayhtml/doc/theory/theory.smarts.html  
https://daylight.com/dayhtml_tutorials/languages/smarts/smarts_examples.html

In [ ]:
# pf = propfit.PropFit()
# pf = propfit.PropFit(filename='default_database_with_aq_props.csv')
pf = propfit.PropFit(filename='default_database_with_aq_props.csv', props=['Gh','Hh','Cph','V','Hig','Sig','Cpig','Gaq','Haq','Cpaq'])

### 3. Combine molecular properties with group data using the dataprep() function  
This step will take the input file with molecular thermodynamic properties and combine it with the group matching for each molecule.  
Default arguments
- average = True (average values when their is more than one measurement for a molecular property. 
- order = 2 (1st or 2nd order method)
- output_name = None (if no name is supplied, the file will be called 'properties and groups.csv'

This cell will take a minute.

In [ ]:
pf.dataprep()

##### 3.1. Visualize the file you just generated

In [ ]:
df = pd.read_csv('properties and groups.csv')
df.head()

### 4. Regress thermodynamic properties of groups  
Using the file you just generated, regress the thermodynamic properties of groups in the dataframe using the group_property_estimator() function. 
- filename = Name of the file containing compounds, properties, and group matching data.
- props = ['Gh','Hh','Cph','V','Hig','Sig','Cpig'] (list of group properties that need to be regressed - these need to match the column names in the input file)
- ignore = list of properties you do not wish to regress.

In [ ]:
pf.group_property_estimator('properties and groups.csv', props = ['Gh','Hh','Cph','V','Hig','Sig','Cpig','Gaq','Haq','Cpaq'])

### 5. Check for overfitting of the model  
Compare how well the model estimates properties of the training set versus the test set. The default training size is 80% while the default test size is 20%, so it is typical for the errors in the test set to be higher. However, is the gap between the training and test set errors increases, be wary that the model is overfitting to the training set. This cell will take the longest to run, but should be run once to determine if it is reasonble to use the group data for new molecular estimations. It will produce a bar plot of training and test set errors for each property. A dataframe with the values will also be produced, named 'stats df.csv'.  
- repeats = how many iterations to do the semi-random train-test-split. Default: 100
- test_size = what fraction of the entire database to make the test set. Default: 0.2
- filename = name of the CSV with properties and groups from which to perform the train-test-split
- output_name = name of output file
- show = True or False. Show plot of errors?

In [ ]:
# pf.tts()

### 6. Generate the group properties datasheet  
Once you are satisfied with the data, use the generate() function to make a dataframe which contains thermodynamic properties of each group. These dataframes can be used as input to the Estimate() function of the sister package, AqOrg.

- filename = 'properties and groups regressed'
- order: order of approximation
- hyd_props = ['Gh','Hh','Cph','V'] (list of hydration properties you want to regress)
- gas_props = ['Hig','Sig','Cpig'] (list of ideal gas properties you want to regress)

In [ ]:
pf.generate()

### 7. Estimate properties of new molecules  
Provide the name of the dataframes you just generated to the Estimate function of AqOrg to estimate properties of new molecules. Be careful not to re-estimate properties for molecules that have already been experimentally determined. 

In [ ]:
gas_props = Estimate(name="phytanol", state="gas", ig_method="custom", group_data="gas props.csv")

In [ ]:
aq_props = Estimate(name="phytanol", state="aq", group_data="hyd props.csv",
                    Gig = gas_props.Gig,
                    Hig = gas_props.Hig,
                    Sig = gas_props.Sig, 
                    Cpig = gas_props.Cpig
                   )

Add your new data into pyCHNOSZ (Boyer, 2025), a python port of the thermodynamic modeling R software CHNOSZ (Dick, 2019), using 'add_OBIGT(aq_props.OBIGT)'.

In [ ]:
add_OBIGT(aq_props.OBIGT)

View the molecule with 'info(info('molecule'))'.

In [ ]:
info(info('phytanol'))

### References

Boyer, G., Robare, J., & Shock, E. (2025). AqOrg : Python package for estimating thermodynamic properties of aqueous organic molecules (v0.2.1). Zenodo. https://doi.org/10.5281/zenodo.14963369  
Boyer, G. (2025). pyCHNOSZ: Python port for the thermodynamic package CHNOSZ (v1.0.0). Zenodo. https://doi.org/10.5281/zenodo.11406142
Dick, J.M. (2019). CHNOSZ: Thermodynamic calculations and diagrams for geochemistry. Frontiers in Earth Science, 7, p.180. https://doi.org/10.3389/feart.2019.00180